# KCT demo: from SharePoint folder to verdict

**For the presenter.** Run all cells before the meeting (Run All), then walk through them top to bottom; every step has a short *Say:* note. Nothing needs typing during the demo.

| Mode | When | Set in the first cell |
|---|---|---|
| `replay` | **The demo.** Real Gemini answers recorded earlier, played back: no network, no waiting, no cost | `MODE = "replay"` (default when a recording exists) |
| `live` | Show it running against Gemini now (needs the Google project) | `MODE = "live"` |
| `fake` | Without any AI: fixed sample answers, for trying the notebook | `MODE = "fake"` |

To make the replay recording once: set `MODE = "live"` and `RECORD = True`, run all, and the answers are saved to `demo/recordings/demo/`. After changing `MODE`, restart the kernel (Kernel > Restart).

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

MODE = "auto"          # auto | replay | live | fake   (auto: replay when a recording exists, else fake)
RECORD = False         # live only: save the answers for replay
MANUAL_HOURS = None    # your estimate: hours to test this control by hand today (shown in step 10)
HOURLY_RATE = None     # optional: cost of a tester hour, to show the saving in money

from services.config import load_env_file
load_env_file()
recording = PROJECT / "demo" / "recordings" / "demo"
if MODE == "auto":
    MODE = "replay" if recording.is_dir() and any(recording.iterdir()) else "fake"
if MODE == "replay":
    os.environ.update(KCT_AI_PROVIDER="replay", KCT_AI_REPLAY="demo")
elif MODE == "fake":
    os.environ["KCT_AI_PROVIDER"] = "fake"
elif RECORD:
    os.environ["KCT_AI_RECORD"] = "demo"

from services.app import boot
from services.config import settings
from services.presentation import notebook as show

kct = boot()
show.quiet()
config = settings()
display(show.banner("Key Control Testing with AI", "From the SharePoint control folder to a traceable verdict per test step"))
show.mode(config.ai_provider, config.ai_model, config.ai_replay)

In [ ]:
show.step(1, "The input: a control folder from SharePoint",
          "Exactly as the team keeps it today: control details and test plan, and the evidence collected for testing.")

*Say:* "This is the folder as it sits in SharePoint today. Nothing new to learn for the team: control details and test plan in one folder, evidence in the other. Word, Excel, Outlook emails, PDFs."

In [ ]:
from factories import ControlFolderFactory

demo_root = config.storage_root / "demo"
folder = ControlFolderFactory(excel_rows=100, pdf_pages=30).create(demo_root / "sharepoint")
zip_path = demo_root / "control.zip"
zip_path.write_bytes(ControlFolderFactory(excel_rows=100, pdf_pages=30).zip())
show.tree(folder)

In [ ]:
show.step(2, "Import: the files are recognised and sorted",
          "Downloaded from SharePoint as one ZIP. The AI tells control details from test plan; the ZIP is kept as it was received.")

In [ ]:
result = kct.importer.import_zip(zip_path)
show.sorting(result)

*Say:* "The original ZIP is stored untouched, the files are sorted, and every decision has a reason. A file in a format the AI can't read is listed, not silently dropped."

In [ ]:
show.step(3, "Understanding the control: test steps",
          "The AI reads the control details and the test plan and writes the test steps a tester would follow, each with clear pass and fail criteria.")

In [ ]:
interpretation = kct.interpretations.generate(result.case_id)
show.test_steps(interpretation)

In [ ]:
step_0472_04 = next(s for s in interpretation.test_steps if s.reference == "0472-04")
show.criteria(step_0472_04)

*Say:* "Every step mirrors the test plan, and the AI adds steps for requirements in the control details that the plan doesn't cover (DERIVED-…). Each step says exactly when it passes and when it fails, so it can be checked against evidence."

In [ ]:
show.step(4, "The evidence, as the AI sees it",
          "Word becomes PDF, Excel becomes a table that keeps the red rows and comments, emails keep the whole thread and their attachments.")

In [ ]:
case = kct.evidences.case(result.case_id)
units = case.units()
excel = next(u for u in units if u.kind == "excel")
show.excel_preview(excel.groups[0][0].data.decode("utf-8"))

In [ ]:
email = next(u for u in units if u.kind == "email")
show.markdown(email.anchor.data.decode("utf-8"), max_lines=45)

*Say:* "A plain export would lose the colors: here the red rows marked by the preparer, the reviewer's comment and the hidden column all reach the AI. The email keeps its full thread, the forwarded approval and the screenshots."

In [ ]:
show.step(5, "The assessment",
          "Each evidence file (big ones in parts) is checked against every test step, then all findings are combined into one verdict per step.")

In [ ]:
progress = show.Progress("Assessing the evidence against the test steps")
assessment = kct.assessments.assess(result.case_id, progress=progress)

In [ ]:
show.step(6, "The result", "One verdict per test step.")

In [ ]:
show.kpis(assessment, config.price_currency)

In [ ]:
show.matrix(assessment)

*Say:* "This is the picture behind every verdict. Green: the file shows the step is met. Red: the file shows it fails. A step is only effective when the evidence supports it, so an empty row means no evidence and not effective."

In [ ]:
show.step(7, "Why a step is not effective", "The exceptions, with exactly where they are in the evidence.")

In [ ]:
steps = {s.id: s for s in interpretation.test_steps}
for verdict in assessment.verdicts:
    if verdict.verdict != "effective":
        display(show.verdict_card(verdict, steps.get(verdict.test_step_id)))

*Say:* "0472-04: two resets were done before the identity check, by a different agent. The AI found one in the ticket export and one in the audit log, where the preparer had marked it red, and it tells us the exact page and row."

In [ ]:
show.step(8, "Why it can be trusted", "The guardrails, with this run's numbers.")

In [ ]:
show.trust(assessment, runs=len(kct.assessments.index(result.case_id)))

In [ ]:
show.step(9, "The report", "A read-only PDF with an executive summary, every verdict, the exceptions and the evidence fingerprints.")

In [ ]:
report_path = (config.storage_root / assessment.metadata.report_path) if config.files_driver == "local" else assessment.metadata.report_path
show.report_link(str(report_path), assessment.metadata.report_sha256 or "")

In [ ]:
show.step(10, "Time", "What a tester gets back.")

In [ ]:
show.savings(assessment, MANUAL_HOURS, HOURLY_RATE, config.price_currency)

*Say:* "The tester still decides: KCT prepares the verdicts with the evidence behind them, the tester reviews. Next step: a pilot on real controls in the bank's own Google project."

---
**After the demo, for questions**
- Same thing through the API: `POST /api/v1/cases/import-zip`, then `POST /api/v1/cases/{case}/assessments`.
- Every step on its own: notebooks `notebooks/07` to `12`.
- How it works and its limits: `OVERVIEW.md`.